# Otomotif: CAN, ISO-TP, UDS, dan OBD-II

Scan tool dan ECU mesin simulasi berbagi bus CAN virtual. Ganti URI menjadi `socketcan:can0` atau `slcan:COM5` untuk perangkat sungguhan — hanya pada kendaraan yang Anda berwenang servis.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.6.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.6.0-preview.1"
#r "nuget: IoTCom.Net.Protocols.Uds, 0.6.0-preview.1"

## Frame CAN dalam notasi candump
Frame divalidasi (identifier 11/29-bit, panjang klasik atau CAN FD).

In [ ]:
using IoTCom.Net.Transport.Can;

var request = CanFrame.Parse("7DF#02010C");   // OBD-II: engine rpm
Console.WriteLine($"{request} → id 0x{request.Id:X3}, {request.Data.Length} bytes, DLC {request.Dlc}");
var fd = new CanFrame(0x18DAF110, CanDlc.Pad(new byte[20]), CanFrameFlags.Extended | CanFrameFlags.Fd);
Console.WriteLine($"CAN FD: {fd.Data.Length} bytes (padded), DLC {fd.Dlc}");

## Simulator ECU dan scan tool di bus virtual

In [ ]:
using IoTCom.Net.Protocols.Uds;

var net = new VirtualCanNetwork();
await using var ecu = EcuSimulator.Create(net.CreateNode());
await ecu.StartAsync();
var bus = net.CreateNode();
await bus.ConnectAsync();
using var sniffer = bus.OpenReader();

await using var obd = ObdClient.Create(bus);
await obd.ConnectAsync();
foreach (var pid in new[] { ObdPids.EngineRpm, ObdPids.CoolantTemperature, ObdPids.ModuleVoltage })
    Console.WriteLine(await obd.ReadPidAsync(pid));
Console.WriteLine($"VIN {await obd.ReadVinAsync()}");

## ISO-TP di jalur
VIN 17 karakter butuh first frame, flow control, dan consecutive frame — dipecah oleh state machine Rust.

In [ ]:
while (sniffer.TryRead(out var f)) if (f.Data.Length > 0 && f.Data.Span[0] >> 4 is 1 or 2 or 3) Console.WriteLine(f);

## UDS: identifikasi, kode kerusakan, dan security access

In [ ]:
await using var uds = UdsClient.Create(bus);
await uds.ConnectAsync();
Console.WriteLine(await uds.ReadStringAsync(UdsDid.SoftwareVersion));
foreach (var dtc in await uds.ReadDtcsAsync()) Console.WriteLine($"{dtc}  {dtc.Status}");

try { await uds.WriteDataByIdentifierAsync(UdsDid.RepairShopCode, "WS-01"u8.ToArray()); }
catch (UdsNegativeResponseException ex) { Console.WriteLine(ex.Message); }

await uds.StartSessionAsync(UdsSession.Extended);
await uds.SecurityAccessAsync(0x01, EcuSimulator.ComputeKey);
await uds.WriteDataByIdentifierAsync(UdsDid.RepairShopCode, "WS-01"u8.ToArray());
Console.WriteLine(await uds.ReadStringAsync(UdsDid.RepairShopCode));

## Lebih lanjut
Demo Galeri *Diagnostik kendaraan* menampilkan tumpukan yang sama dengan dasbor langsung; CLI punya `iotcom obd live --can sim`. Lihat `docs/id/protocols/uds.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*